# MicroDuck × MuJoCo on Google Colab — 動画保存から強化学習まで

[Microduck](https://github.com/pollen-robotics/microduck)（Pollen Robotics の約 800 g・高さ 25 cm の二足ロボット）を
Google Colab 上の MuJoCo で動かして動画を保存し、そのあと [microduck_rl](https://github.com/pollen-robotics/microduck_rl)
で強化学習（PPO）を試すノートブックです。上から順に実行してください。

**最初に:** メニュー「ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU」を選びます。
Phase 1（素の MuJoCo）は GPU なしでも動きますが、Phase 2 の学習は **NVIDIA GPU が必須** です（T4 で動作確認程度、本格的な学習は L4 / A100 推奨）。

| 章 | 内容 | GPU | 目安時間 |
|---|---|---|---|
| 0 | 概要と用語 | – | – |
| 1 | 環境準備（uv・リポジトリ・依存関係） | – | 5〜10 分（初回の `uv sync` が数 GB） |
| 2 | Phase 1: 素の MuJoCo で動かして動画保存 | 不要 | 数分 |
| 3 | Phase 1 応用: 学習済みポリシーで歩かせる | 不要 | 数分 |
| 4 | Phase 1b: mjlab 学習環境のスモークテスト | 必須 | 数分 |
| 5 | Phase 2: 強化学習 | 必須 | 数十分〜数時間 |
| 6 | 評価・動画・ONNX エクスポート | 必須 | 数分 |
| 7 | 次の実験 / トラブルシューティング / 付録 | – | – |

> **検証状況（2026-10-04 時点、microduck_rl `1f2e4af`）**
> - 2・3 章のスクリプトと、学習 → チェックポイント → `play --video True` → `export.py` → ONNX 再生の一連の流れは、
>   macOS（CPU、`--gpu-ids None`、極小規模）で動作を確認済み。CLI オプションは mjlab 1.3.0 の `--help` と照合済み。
> - **Colab（Linux + CUDA）上での実行と、GPU での学習時間は未検証** です。うまくいかない場合は 7 章のトラブルシューティングを参照してください。

## 0. 概要と用語

```
pollen-robotics/microduck      … 実機のランタイム（Rust）。学習済み ONNX ポリシーをロボット上で動かす
pollen-robotics/microduck_rl   … シミュレーション/学習側（Python）。ロボットの MJCF モデルと mjlab の学習タスクを持つ  ← このノートで使う
        └─ mjlab (MuJoCo Warp + rsl_rl PPO) で学習 → ONNX にエクスポート → 実機へ
```

| 用語 | 意味 |
|---|---|
| **MuJoCo** | DeepMind の物理シミュレータ。`mujoco` パッケージは CPU で動く |
| **MJCF** | MuJoCo のモデル記述形式（XML）。リンク・関節・アクチュエータ・衝突形状・メッシュ・床・初期姿勢などを書く。MicroDuck のものは CAD（Onshape）から自動生成されている |
| **keyframe** | MJCF に保存された名前付きの姿勢。`scene.xml` には `INIT` / `STAND` / `SIT` / `FOLD` がある |
| **ヘッドレス / EGL** | 画面（ディスプレイ）なしで描画すること。Colab には画面がないので、`MUJOCO_GL=egl` で GPU のオフスクリーン描画を使う |
| **mjlab** | MuJoCo Warp（GPU 版 MuJoCo）で数千の環境を並列に動かす強化学習フレームワーク。`train` / `play` コマンドを提供 |
| **PPO** | 学習アルゴリズム（Proximal Policy Optimization）。rsl_rl の実装を使う |
| **ONNX** | 学習済みポリシーの保存形式。実機もこの形式を読み込む |
| **viser** | ブラウザで見られる 3D ビューア。mjlab の `play` は画面がないとき自動でこれを使う |

**GUI が必要なもの・不要なもの**
- `scripts/infer_policy.py`（キーボードで歩かせるツール）は `mujoco.viewer.launch_passive` で **デスクトップのウィンドウを開く** ため Colab では使えません。
  キー入力は端末（TTY）から読む仕組みで、TTY がないと自動で無効になるだけなので、問題はウィンドウの方です。→ 3 章で「ウィンドウなし版」を用意します。
- `train`（学習）・`play --video True`（評価動画）・`scripts/export.py`・`--hf-jobs`（Hugging Face Jobs での学習）は **画面なしで動きます**。

## 1. 環境準備

### 1-1. GPU と Python の確認
`nvidia-smi` で GPU 名が表示されれば OK です（Phase 1 だけなら GPU なしでも可）。
microduck_rl は Python 3.12 を要求しますが、足りなければ `uv` が自動で 3.12 を取得するので Colab の Python のバージョンは気にしなくて大丈夫です。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU なし（Phase 1 のみ実行可能）"
!python --version

### 1-2. ツールと環境変数
- `uv`: microduck_rl が使う Python パッケージマネージャ
- `UV_HTTP_TIMEOUT=600`: 初回の依存関係ダウンロード（CUDA 入りで数 GB）がタイムアウトしないように
- `MUJOCO_GL=egl`: 画面なしでの描画に EGL を使う（GPU ランタイム向け。CPU ランタイムでは `osmesa` に変えて `apt-get install -y libosmesa6` が必要）

In [ ]:
%env UV_HTTP_TIMEOUT=600
%env MUJOCO_GL=egl
%env PYOPENGL_PLATFORM=egl
!pip -q install uv
!uv --version

### 1-3. microduck_rl の取得と依存関係のインストール
このノートを検証したコミット（`1f2e4af`）に固定しています。最新版を使いたい場合は `COMMIT = None` にしてください。
`uv sync` は初回 5〜10 分ほどかかります。

In [ ]:
import os
COMMIT = "1f2e4afcf33fbc75172b893c82fb66da9e3fc7ea"  # None にすると最新の main

%cd /content
if not os.path.isdir("microduck_rl"):
    !git clone https://github.com/pollen-robotics/microduck_rl
%cd /content/microduck_rl
if COMMIT:
    !git checkout -q {COMMIT}
!git log --oneline -1
!uv sync

登録されている MicroDuck のタスク一覧を表示します（`Mjlab-Velocity-Flat-MicroDuck` などが出れば成功）。

In [ ]:
!uv run list-envs 2>/dev/null | grep MicroDuck

### 1-4. （任意）学習ログを Google Drive に保存する
Colab のセッションは数時間で切れ、`/content` の中身は消えます。学習のチェックポイントを残したい場合は `USE_DRIVE = True` にしてください。
mjlab 1.3.0 はログの保存先が `logs/rsl_rl/<実験名>/<日時>_<run名>/` に固定（変更オプションなし）なので、
`logs` フォルダ自体を Drive へのシンボリックリンクにします。

In [ ]:
import os, shutil
USE_DRIVE = False
DRIVE_LOGS = "/content/drive/MyDrive/microduck_logs"

os.chdir("/content/microduck_rl")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_LOGS, exist_ok=True)
    if os.path.isdir("logs") and not os.path.islink("logs"):
        shutil.copytree("logs", DRIVE_LOGS, dirs_exist_ok=True)
        shutil.rmtree("logs")
    if not os.path.islink("logs"):
        os.symlink(DRIVE_LOGS, "logs")
print("logs ->", os.path.realpath("logs"))

## 2. Phase 1: 素の MuJoCo で動かして動画を保存する

まずは強化学習なしで、MJCF モデル（`src/mjlab_microduck/robot/microduck/scene.xml`）を `mujoco` で読み込み、
`STAND` 姿勢で立たせて首を振らせる動画と、keyframe ごとの静止画を保存します。

ポイント:
- 描画は `mujoco.Renderer`（オフスクリーン）で行い、`mediapy.write_video` で mp4 にします。カメラは胴体（`trunk_base`）を追いかけます。
- MJCF の位置アクチュエータは `kp=0.55`（実機サーボ相当で弱い）なので、**ポリシーなしで関節を固定しても 1 秒ほどで後ろに倒れます**。
  そこで既定では `kp=2.0`（`infer_policy.py` の待機モードと同じ値）に上げて立たせています。
- 14 個の関節: 0–4 左脚（hip_yaw, hip_roll, hip_pitch, knee, ankle）、5–8 首/頭（neck_pitch, head_pitch, head_yaw, head_roll）、9–13 右脚。

In [ ]:
%%writefile render_stand.py
"""MicroDuck を素の MuJoCo で動かし、オフスクリーン描画で mp4 / png を保存する。

microduck_rl リポジトリのルートで実行する:
    uv run python render_stand.py                              # kp=2.0 で立たせる
    uv run python render_stand.py --kp 0.55 --out fall.mp4     # XML 既定の kp では倒れる
"""
import argparse

import mujoco
import mediapy
import numpy as np

SCENE = "src/mjlab_microduck/robot/microduck/scene.xml"
WIDTH, HEIGHT, FPS = 640, 480, 30
DURATION_S = 6.0

parser = argparse.ArgumentParser()
# XML の位置アクチュエータは kp=0.55（実機 XL330 相当で弱い）。ポリシーなしで姿勢を保持すると 1 秒ほどで倒れるので、
# 既定では infer_policy.py の待機モード (_STANDBY_KP = 2.0) と同じ値に上げて「立たせて」おく。
parser.add_argument("--kp", type=float, default=2.0, help="位置アクチュエータのゲイン (XML 既定は 0.55)")
parser.add_argument("--out", default="stand.mp4")
args = parser.parse_args()
KP = args.kp

model = mujoco.MjModel.from_xml_path(SCENE)
model.actuator_gainprm[:, 0] = KP   # 位置アクチュエータ: force = kp * (ctrl - q)
model.actuator_biasprm[:, 1] = -KP
data = mujoco.MjData(model)
print(f"bodies={model.nbody} joints={model.njnt} actuators={model.nu} timestep={model.opt.timestep}")
print("keyframes:", [model.key(i).name for i in range(model.nkey)])
print("actuators:", [model.actuator(i).name for i in range(model.nu)])

# トラッキングカメラ: 胴体 (trunk_base) を追いかける
camera = mujoco.MjvCamera()
camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
camera.trackbodyid = model.body("trunk_base").id
camera.distance = 0.6
camera.azimuth = 135
camera.elevation = -20

renderer = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)

# --- 1) STAND 姿勢で立たせ、首と頭を正弦波で動かす動画 ---
mujoco.mj_resetDataKeyframe(model, data, model.key("STAND").id)
stand_ctrl = data.ctrl.copy()
head_ids = [model.actuator(n).id for n in ("neck_pitch", "head_pitch", "head_yaw", "head_roll")]

frames = []
steps_per_frame = int(round(1.0 / (FPS * model.opt.timestep)))
for _ in range(int(DURATION_S * FPS)):
    t = data.time
    data.ctrl[:] = stand_ctrl
    if t > 1.0:  # 最初の 1 秒は静止して接地させる
        data.ctrl[head_ids[2]] += 0.6 * np.sin(2 * np.pi * 0.5 * t)   # head_yaw: 左右を見る
        data.ctrl[head_ids[1]] += 0.3 * np.sin(2 * np.pi * 0.25 * t)  # head_pitch: うなずく
    for _ in range(steps_per_frame):
        mujoco.mj_step(model, data)
    renderer.update_scene(data, camera)
    frames.append(renderer.render())

mediapy.write_video(args.out, frames, fps=FPS)
print(f"saved {args.out} ({len(frames)} frames), trunk z = {data.qpos[2]:.3f} m")

# --- 2) keyframe ごとの静止画 (INIT / STAND / SIT / FOLD) ---
images = []
for i in range(model.nkey):
    mujoco.mj_resetDataKeyframe(model, data, i)
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera)
    images.append(renderer.render())
mediapy.write_image("keyframes.png", np.concatenate(images, axis=1))
print("saved keyframes.png")

In [ ]:
!uv run python render_stand.py

In [ ]:
from IPython.display import Video, Image, display
display(Video("stand.mp4", embed=True, width=640))
display(Image("keyframes.png"))  # 左から INIT / STAND / SIT / FOLD

比較: XML 既定のゲイン（`kp=0.55`）のままだとどうなるか。**倒れるのが正常** です。
このロボットは関節を固めるだけでは立っていられないので、バランスを取るポリシー（＝強化学習）が必要になります。

In [ ]:
!uv run python render_stand.py --kp 0.55 --out fall.mp4
display(Video("fall.mp4", embed=True, width=640))

動画をパソコンに保存するには次のセルを実行します（Drive に保存したい場合は `shutil.copy` で `/content/drive/MyDrive/` へコピー）。

In [ ]:
from google.colab import files
files.download("stand.mp4")

## 3. Phase 1 応用: 学習済みポリシーで歩かせる（ウィンドウなし版 infer_policy）

Pollen Robotics が公開している学習済みポリシー（[pollen-robotics/microduck-policies](https://huggingface.co/pollen-robotics/microduck-policies)、
ロボット出荷時と同じもの）を CPU の MuJoCo で動かして動画にします。

`headless_walk.py` は `scripts/infer_policy.py` の部品（観測ベクトルの組み立て `PolicyInference`、
実機サーボ XL330 を電圧レベルで再現する BAM アクチュエータモデル `load_mujoco_with_bam`）を **そのまま import して再利用** し、
ウィンドウとキー入力のループだけを `mujoco.Renderer` に置き換えたものです。速度指令は引数で固定します。

- `velstand.onnx`: 現在の標準の歩行ポリシー（指令 0 では立ち止まる）
- `alpha_walking.onnx`: 旧歩行ポリシー
- 観測は 61 次元（角速度 3 + 重力方向 3 + 関節角 14 + 関節速度 14 + 前回の行動 14 + 指令 13）、出力は 14 関節の目標角、50 Hz 制御
- 指令の学習範囲: 前後 ±0.3 m/s、左右 ±0.2 m/s、旋回 ±1.5 rad/s。**0.15 m/s 程度だとほぼ足踏み** で、0.3 m/s で 8 秒に約 0.7 m 進みます（macOS で確認）

In [ ]:
from huggingface_hub import hf_hub_download
for f in ["velstand.onnx", "alpha_walking.onnx"]:
    print(hf_hub_download("pollen-robotics/microduck-policies", f, local_dir=".", token=False))

In [ ]:
%%writefile headless_walk.py
"""学習済み ONNX ポリシーを GUI なし (mujoco.Renderer) で再生し mp4 に保存する。

scripts/infer_policy.py の main() から viewer とキー入力を取り除いたもの。
観測の組み立て・BAM アクチュエータ・初期姿勢はすべて infer_policy.py の部品をそのまま使う。

microduck_rl リポジトリのルートで実行する:
    uv run python headless_walk.py --onnx velstand.onnx --lin-vel-x 0.3 --seconds 10 --out walk.mp4
"""
import argparse
import sys
import types

# infer_policy.py は先頭で `import mujoco.viewer`（GLFW ウィンドウ用）を行う。
# 画面のない環境でも読み込めるよう、使わない viewer モジュールを空のダミーに差し替えておく。
sys.modules.setdefault("mujoco.viewer", types.ModuleType("mujoco.viewer"))
sys.path.insert(0, "scripts")

import mediapy  # noqa: E402
import mujoco  # noqa: E402
import numpy as np  # noqa: E402

import infer_policy as ip  # noqa: E402

parser = argparse.ArgumentParser()
parser.add_argument("--onnx", required=True, help="61 次元観測の歩行ポリシー (export.py の出力や HF の velstand.onnx)")
parser.add_argument("--lin-vel-x", type=float, default=0.3, help="前後速度指令 m/s (学習範囲 ±0.3)")
parser.add_argument("--lin-vel-y", type=float, default=0.0, help="左右速度指令 m/s (学習範囲 ±0.2)")
parser.add_argument("--ang-vel-z", type=float, default=0.0, help="旋回速度指令 rad/s (学習範囲 ±1.5)")
parser.add_argument("--seconds", type=float, default=10.0)
parser.add_argument("--out", default="walk.mp4")
parser.add_argument("--width", type=int, default=640)
parser.add_argument("--height", type=int, default=480)
args = parser.parse_args()

# --- infer_policy.main() と同じ初期化 (BAM 有効・既定電圧) ---
bam_model = ip.load_bam_model(ip.BAM_KP_FW, 7.4, 0.0)
model, data, bam_ctrl, _ = ip.load_mujoco_with_bam(ip.MICRODUCK_XML, bam_model, 0.005, 0.1, ip.BAM_VIN_MIN)
policy = ip.PolicyInference(model, data, walking_onnx_path=args.onnx, bam_ctrl=bam_ctrl,
                            use_projected_gravity=True, new_cmd_obs=True)
policy.set_vel_cmd(args.lin_vel_x, args.lin_vel_y, args.ang_vel_z)

qpos_adr = model.jnt_qposadr[model.joint("trunk_base_freejoint").id]
data.qpos[qpos_adr:qpos_adr + 3] = [0.0, 0.0, 0.125]
data.qpos[qpos_adr + 3:qpos_adr + 7] = [1, 0, 0, 0]
for i, qpos_idx in enumerate(policy.joint_qpos_indices):
    data.qpos[qpos_idx] = policy.default_pose[i]
bam_ctrl.reset(data.qpos)
policy.set_position_targets(policy.default_pose)
mujoco.mj_forward(model, data)
assert policy.get_observations().size == 61, "61 次元観測のポリシーではありません"

# --- 50 Hz 制御ループ (decimation 4 × 5 ms)、2 制御ステップごとに 1 フレーム = 25 fps ---
camera = mujoco.MjvCamera()
camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
camera.trackbodyid = model.body("trunk_base").id
camera.distance, camera.azimuth, camera.elevation = 0.7, 135, -20
renderer = mujoco.Renderer(model, height=args.height, width=args.width)

decimation, frame_every = 4, 2
control_dt = decimation * model.opt.timestep
frames = []
for step in range(int(args.seconds / control_dt)):
    policy.apply_action(policy.infer())
    for _ in range(decimation):
        bam_ctrl.update()
        mujoco.mj_step(model, data)
    if step % frame_every == 0:
        renderer.update_scene(data, camera)
        frames.append(renderer.render())

mediapy.write_video(args.out, frames, fps=round(1 / (control_dt * frame_every)))
x, y, z = data.qpos[qpos_adr:qpos_adr + 3]
print(f"saved {args.out}: {len(frames)} frames, final trunk pos x={x:.2f} y={y:.2f} z={z:.3f} m")

In [ ]:
!uv run python headless_walk.py --onnx velstand.onnx --lin-vel-x 0.3 --seconds 10 --out walk.mp4
display(Video("walk.mp4", embed=True, width=640))

指令を変えて試してみましょう（例: その場で旋回、横歩き）。

In [ ]:
!uv run python headless_walk.py --onnx velstand.onnx --lin-vel-x 0 --ang-vel-z 1.0 --seconds 8 --out turn.mp4
!uv run python headless_walk.py --onnx alpha_walking.onnx --lin-vel-x 0 --lin-vel-y 0.2 --seconds 8 --out side.mp4
display(Video("turn.mp4", embed=True, width=480), Video("side.mp4", embed=True, width=480))

## 4. Phase 1b: mjlab 学習環境のスモークテスト（GPU 必須）

ここから GPU を使います。本番の学習の前に、少数の環境・数イテレーションだけ学習して
「GPU シミュレーション → 学習 → チェックポイント保存 → 学習中の動画保存」が通るかを確認します。

メモ:
- mjlab 1.3.0 の `--video` などの真偽値オプションは **`--video True` のように値を書く** 形式です（`--video` だけではエラー）。
- `--agent.logger tensorboard` にすると Weights & Biases（wandb）のアカウントなしで動きます（既定は `wandb`）。
- `play --agent zero/random` は動画を保存できない仕様（チェックポイントが必要）なので、動作確認はこの小さな学習で行います。

In [ ]:
cmd = ("uv run train Mjlab-Velocity-Flat-MicroDuck"
       " --env.scene.num-envs 64 --agent.max-iterations 3 --agent.save-interval 1"
       " --agent.logger tensorboard --agent.run-name smoke"
       " --video True --video-length 100 --video-interval 50")
!{cmd} 2>&1 | grep -vi warn | tail -n 25

In [ ]:
import glob, os
from IPython.display import Video, display
smoke = sorted(glob.glob("logs/rsl_rl/velocity/*_smoke"))[-1]
print(smoke); print(sorted(os.listdir(smoke)))
vids = sorted(glob.glob(f"{smoke}/videos/train/*.mp4"))
print(vids)
if vids:
    display(Video(vids[0], embed=True, width=640))  # 学習前のポリシーなので転ぶのが正常

## 5. Phase 2: 強化学習

主タスク `Mjlab-Velocity-Flat-MicroDuck`（平地で速度指令と頭の向きの指令に従って歩く）を学習します。

**規模の目安**（README より: 4096 環境で 1〜2 時間学習すると実用的な歩容になる。設定上の上限は 50,000 イテレーション）
| GPU | `NUM_ENVS` | `MAX_ITERS` の例 | メモ |
|---|---|---|---|
| T4 (16 GB) | 1024〜2048 | 500〜1000 | お試し程度。速度・メモリとも未検証 |
| L4 / A100 | 4096 | 1500〜3000 | README の推奨設定 |

学習は **バックグラウンドで実行** し、ログを `train.log` に書き出します（セルの出力が膨大になってブラウザが重くなるのを防ぐため）。
進み具合は次のセルや TensorBoard で確認します。チェックポイントは 250 イテレーションごと（`--agent.save-interval`）に保存されます。

**wandb を使う場合:** `LOGGER = "wandb"` にして、下のセルで `wandb login` を実行してください（プロジェクト名は `mjlab_microduck`）。

In [ ]:
TASK = "Mjlab-Velocity-Flat-MicroDuck"
NUM_ENVS = 4096        # T4 なら 1024〜2048 に下げる
MAX_ITERS = 1500
LOGGER = "tensorboard" # "wandb" も可
RUN_NAME = "colab"

if LOGGER == "wandb":
    !uv run wandb login

In [ ]:
train_cmd = (f"uv run train {TASK}"
             f" --env.scene.num-envs {NUM_ENVS} --agent.max-iterations {MAX_ITERS}"
             f" --agent.logger {LOGGER} --agent.run-name {RUN_NAME}"
             f" --video True --video-length 200")
print(train_cmd)
!nohup {train_cmd} > train.log 2>&1 &
!sleep 60; grep -vi warn train.log | tail -n 15

進み具合の確認（何度でも実行できます）。`Mean reward` が上がり、`Mean episode length` が伸びていけば学習が進んでいます。

In [ ]:
!grep -a "Learning iteration" train.log | tail -n 1
!grep -a -E "Mean reward|Mean episode length|Time elapsed|ETA" train.log | tail -n 4
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/rsl_rl

学習中の動画は、既定で 2000 環境ステップ（約 83 イテレーション）ごとに `videos/train/` に保存されます。最新のものを表示します。

In [ ]:
import glob, os, re
from IPython.display import Video, display
def latest_run(run_name=RUN_NAME, exp="velocity"):
    return sorted(glob.glob(f"logs/rsl_rl/{exp}/*_{run_name}"))[-1]

run_dir = latest_run()
vids = sorted(glob.glob(f"{run_dir}/videos/train/*.mp4"), key=lambda p: int(re.findall(r"(\d+)", os.path.basename(p))[-1]))
print(run_dir, f"{len(vids)} videos")
if vids:
    print(vids[-1]); display(Video(vids[-1], embed=True, width=640))

学習を途中で止めたいときは次のセルを実行します（チェックポイントは残ります）。

In [ ]:
!pkill -f "train {TASK}" && echo stopped || echo "not running"

### セッションが切れたときの再開
`USE_DRIVE = True` でログを Drive に保存していれば、1 章を再実行したあと、最新のチェックポイントから学習を再開できます。
チェックポイント名の正規表現は辞書順で選ばれる（`model_1000.pt` より `model_750.pt` が新しいと判定される）ため、ファイル名を明示して渡します。

In [ ]:
def latest_checkpoint(run_dir):
    pts = glob.glob(f"{run_dir}/model_*.pt")
    return max(pts, key=lambda p: int(re.search(r"model_(\d+)\.pt", p).group(1)))

run_dir = latest_run()
ckpt = latest_checkpoint(run_dir)
print("resume from:", ckpt)
resume_cmd = (train_cmd + " --agent.resume True"
              f" --agent.load-run {os.path.basename(run_dir)} --agent.load-checkpoint {os.path.basename(ckpt)}")
!nohup {resume_cmd} > train.log 2>&1 &

## 6. 評価・動画・ONNX エクスポート

### 6-1. mjlab の `play` で評価動画を撮る
`play` は動画（`--video-length` ステップ分）を `<run>/videos/play/` に保存したあと、**ビューア（Colab では viser）を起動したまま終了しません**。
そこでバックグラウンドで起動し、mp4 ができたら停止します。ついでに viser のビューアを iframe で表示することもできます。

In [ ]:
import subprocess, time, re

run_dir = latest_run()
ckpt = latest_checkpoint(run_dir)
print("checkpoint:", ckpt)
play_log = open("play.log", "w")
proc = subprocess.Popen(
    ["uv", "run", "play", TASK, "--agent", "trained", "--checkpoint-file", ckpt,
     "--num-envs", "1", "--video", "True", "--video-length", "400", "--viewer", "viser"],
    stdout=play_log, stderr=subprocess.STDOUT)

for _ in range(120):  # 最大 10 分待つ
    mp4s = glob.glob(f"{run_dir}/videos/play/*.mp4")
    if mp4s:
        break
    time.sleep(5)
time.sleep(5)
print(mp4s)
if mp4s:
    display(Video(mp4s[-1], embed=True, width=640))

（任意）viser ビューアをこのノート内に表示します。マウスで視点を動かせます。見終わったら次のセルで `play` を止めてください。

In [ ]:
from google.colab import output
m = re.search(r"listening \*:(\d+)", open("play.log").read())
port = int(m.group(1)) if m else 8080
print("viser port:", port)
output.serve_kernel_port_as_iframe(port, height=600)

In [ ]:
proc.terminate(); proc.wait(timeout=30); print("play stopped")

### 6-2. ONNX にエクスポート
実機や `headless_walk.py` で使うには ONNX にします。**必ず `scripts/export.py` を使ってください**
（観測の正規化がグラフに組み込まれます。学習フォルダに自動で出力される `.onnx` や、手作業で変換したものは使わない）。

In [ ]:
!uv run scripts/export.py {TASK} --checkpoint-file {ckpt} --onnx-file my_walk.onnx 2>&1 | tail -n 1

### 6-3. 自分のポリシーを CPU の MuJoCo で歩かせる
3 章のスクリプトに自作の ONNX を渡すと、学習時とは別の物理エンジン（CPU 版 MuJoCo + BAM アクチュエータ）で同じポリシーを試せます。
ここでも歩ければ、シミュレータの違いに強いポリシーになっている目安になります。

In [ ]:
from google.colab import files
!uv run python headless_walk.py --onnx my_walk.onnx --lin-vel-x 0.3 --seconds 10 --out my_walk.mp4
display(Video("my_walk.mp4", embed=True, width=640))
files.download("my_walk.onnx")

## 7. 次の実験

**ほかのタスク**（`uv run list-envs` で全一覧）
| タスク ID | 内容 |
|---|---|
| `Mjlab-Velocity-{Flat,Rough}-MicroDuck` | 歩行（このノートで学習したもの）。Rough は凸凹地形 |
| `Mjlab-VelStand-{Flat,Rough}-MicroDuck` | 歩行 + 転倒からの復帰 |
| `Mjlab-StandUp-{Flat,Rough}-MicroDuck` | うつ伏せ・仰向け・座りから立ち上がる |
| `Mjlab-SitStand-{Flat,Rough}-MicroDuck` | 指令で座る ↔ 立つ |
| `Mjlab-GroundPick-{Flat,Rough}-MicroDuck` | かがんで口先で床に触れる |
| `Mjlab-BallKick-Flat-MicroDuck` | ボールを蹴る |
| `Mjlab-Roulade-Flat-MicroDuck` | 前転 |
| `Mjlab-Velocity-Flat-MicroDuck-Rollers` ほか Roller 系 | 足裏の車輪でローラースケート |
| `*-Backlash-MicroDuck` | ギアのガタ（±1°）を模擬したモデルで学習する版（実機への移行に強い） |

`train` の実験名（ログのフォルダ名）はタスクごとに異なります。5 章の `latest_run(exp=...)` の `exp` は `logs/rsl_rl/` の中を見て合わせてください。

**いじれる場所**（`src/mjlab_microduck/`）
- `tasks/microduck_*_env_cfg.py`: 各タスクの設定。先頭の `ENABLE_*` でドメインランダム化（質量・摩擦・電圧などのばらつき）を切り替え
- `tasks/mdp.py`: 報酬・観測・イベントの実装
- `robot/microduck/*.xml`: MJCF モデル
- `AGENTS.md`: 報酬設計で得られた知見のまとめ（環境を作るときの手引き）

**GPU が足りないとき:** 学習コマンドに `--hf-jobs` を付けると Hugging Face Jobs の GPU で学習できます（有料、`scripts/hf/README.md` 参照）。

## 8. トラブルシューティング

| 症状 | 対処 |
|---|---|
| `uv sync` がダウンロード途中で失敗 | `UV_HTTP_TIMEOUT=600` を設定済みか確認して再実行 |
| `error: unrecognized arguments: --video` | 真偽値は `--video True` のように値を書く。`uv run train <TASK> --help` で確認 |
| 描画で `EGL` / `OpenGL` のエラー | GPU ランタイムか確認。CPU ランタイムなら `MUJOCO_GL=osmesa` + `apt-get install -y libosmesa6` |
| `IndexError: list index out of range`（`select_gpus`） | GPU が見えていない。ランタイムを GPU にする（CPU で無理やり試すなら `--gpu-ids None`、非常に遅い） |
| CUDA の out of memory | `NUM_ENVS` を半分にする |
| `play` のセルが終わらない | 仕様（ビューアが起動し続ける）。6-1 のようにバックグラウンドで起動して止める |
| 3 章で `ModuleNotFoundError: infer_policy` | `microduck_rl` のルート（`/content/microduck_rl`）で実行しているか確認 |
| 歩かずに足踏みする | 速度指令が小さすぎる。`--lin-vel-x 0.3` 程度にする |

## 付録: 手元の PC で `infer_policy.py` を使う場合のキー操作

画面のある PC なら `uv run scripts/infer_policy.py --walking velstand.onnx --new-cmd-obs` でウィンドウ付きの対話モードが使えます。
キーは **ウィンドウではなく、起動したターミナルに** 入力します。

| モード | キー | 動作 |
|---|---|---|
| 速度（既定） | ↑ / ↓ | 前後速度 |
| | ← / → | 横移動 |
| | A / E | 旋回 |
| | SPACE | 停止（指令をすべて 0） |
| 共通 | T | ポリシーの一時停止/再開 |
| | P | ランダムに押す（外乱） |
| | Q | 終了 |
| 動作（対応する ONNX を指定したときのみ） | G / Y / K / L / R | 床拾い / 座る・立つ / 左キック / 右キック / 前転 |
| 胴体姿勢（B で切替） | ↑↓ / ←→ / A E / Z S | 高さ / ピッチ / ロール / ヨー |
| 頭（H で切替） | Z S / ↑↓ / ←→ / A E | 首ピッチ / 頭ピッチ / 頭ヨー / 頭ロール |

参考: [microduck_rl README](https://github.com/pollen-robotics/microduck_rl) / [mjlab](https://github.com/mujocolab/mjlab) / [BAM（アクチュエータモデル）](https://github.com/Rhoban/bam)